# MASI Colab Smoke Test

This notebook is for bootstrapping a fresh Colab runtime.

Purpose:
- clone the MASI repo,
- install the required Python packages,
- download the bounded CSJ review slice,
- run the one-click smoke pipeline,
- inspect the output manifest and summaries.

The notebook itself does not reimplement training logic. It uses the repository scripts directly so the Colab run stays consistent with the checked-in codebase.

## How To Use This Notebook

1. Set `REPO_URL` below.
2. Run all cells from top to bottom.
3. Inspect the manifest and summary cells at the end.

If your repo is private, make sure Colab has access to clone it.

In [ ]:
REPO_URL = "https://github.com/<your-user>/<your-repo>.git"
BRANCH = "main"
REPO_DIR = "/content/MASI"
STORAGE_ROOT = "/content/masi_storage"

In [ ]:
%cd /content
!rm -rf "$REPO_DIR"
!git clone --branch "$BRANCH" "$REPO_URL" "$REPO_DIR"
%cd "$REPO_DIR"

In [ ]:
%cd "$REPO_DIR"
!python -m venv .venv
!.venv/bin/pip install --upgrade pip
!.venv/bin/pip install -e ".[recommender]"

In [ ]:
%cd "$REPO_DIR"
!mkdir -p data/raw/amazon_reviews_2023
!mkdir -p "$STORAGE_ROOT"

## Download Bounded CSJ Slice

This uses the repository downloader without `--full-reviews`, so it fetches only a bounded prefix of the CSJ review file for the smoke test.

In [ ]:
%cd "$REPO_DIR"
!PYTHONPATH=src python scripts/download_amazon_csj_dataset.py

In [ ]:
!ls -lh "$REPO_DIR"/data/raw/amazon_reviews_2023

## Run One-Click Smoke Pipeline

This calls the checked-in launcher script with the smoke config.

In [ ]:
%cd "$REPO_DIR"
!PYTHONPATH=src .venv/bin/python scripts/train_masi.py \
  --config configs/masi_train_csj_smoke.json \
  --storage-root "$STORAGE_ROOT"

In [ ]:
!find "$STORAGE_ROOT"/outputs/amazon_csj_smoke_train -maxdepth 3 -type f | sort

In [ ]:
import json
from pathlib import Path

manifest_path = Path(STORAGE_ROOT) / "outputs" / "amazon_csj_smoke_train" / "run_manifest.json"
with manifest_path.open("r", encoding="utf-8") as handle:
    manifest = json.load(handle)

print("Run manifest:", manifest_path)
print("Environment:", manifest["environment"])
print("Token summary path:", manifest["token_summary_path"])
print("Experiment summary path:", manifest["experiment_summary_path"])

In [ ]:
import json
from pathlib import Path

summary_path = Path(STORAGE_ROOT) / "outputs" / "amazon_csj_smoke_train" / "phase3_experiment" / "experiment_summary.json"
with summary_path.open("r", encoding="utf-8") as handle:
    summary = json.load(handle)

high_signal = {
    "mlm_status": summary.get("mlm_status"),
    "generative_finetuning_status": summary.get("generative_finetuning_status"),
    "num_items": summary.get("num_items"),
    "num_train_examples": summary.get("num_train_examples"),
    "num_mlm_examples": summary.get("num_mlm_examples"),
    "warm_metrics": summary.get("warm_metrics"),
    "cold_metrics": summary.get("cold_metrics"),
}
print(json.dumps(high_signal, indent=2))